In [1]:
# ============================================================
# Task 3: Linear Regression — House Price Prediction
# ElevateLabs AI/ML Internship
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150


In [2]:
# ============================================================
# STEP 1 — Load & Explore Dataset
# ============================================================
print("=" * 60)
print("STEP 1: Load & Explore Dataset")
print("=" * 60)

df = pd.read_csv("Housing.csv")
print(f"\n📌 Shape: {df.shape}")
print("\n📌 First 5 rows:")
print(df.head())
print("\n📌 Data Types & Nulls:")
print(df.info())
print("\n📌 Summary Statistics:")
print(df.describe())
print("\n📌 Missing Values:", df.isnull().sum().sum())

STEP 1: Load & Explore Dataset

📌 Shape: (545, 13)

📌 First 5 rows:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

📌 Data Types & Nulls:
<class '

In [3]:
# ============================================================
# STEP 2 — Preprocessing: Encode Categorical Columns
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Preprocessing")
print("=" * 60)

df_model = df.copy()

# Binary yes/no columns → 1/0
binary_cols = ["mainroad", "guestroom", "basement", "hotwaterheating",
               "airconditioning", "prefarea"]
for col in binary_cols:
    df_model[col] = df_model[col].map({"yes": 1, "no": 0})

# furnishingstatus → label encode (furnished=2, semi-furnished=1, unfurnished=0)
furnish_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
df_model["furnishingstatus"] = df_model["furnishingstatus"].map(furnish_map)

print("✅ Binary columns encoded (yes=1, no=0)")
print("✅ furnishingstatus encoded (furnished=2, semi=1, unfurnished=0)")
print("\n📌 Encoded sample:")
print(df_model.head(3))


STEP 2: Preprocessing
✅ Binary columns encoded (yes=1, no=0)
✅ furnishingstatus encoded (furnished=2, semi=1, unfurnished=0)

📌 Encoded sample:
      price  area  bedrooms  bathrooms  stories  mainroad  guestroom  \
0  13300000  7420         4          2        3         1          0   
1  12250000  8960         4          4        4         1          0   
2  12250000  9960         3          2        2         1          0   

   basement  hotwaterheating  airconditioning  parking  prefarea  \
0         0                0                1        2         1   
1         0                0                1        3         0   
2         1                0                0        2         1   

   furnishingstatus  
0                 2  
1                 2  
2                 1  


In [4]:
# ============================================================
# STEP 3 — Part A: Simple Linear Regression (area → price)
# ============================================================
print("\n" + "=" * 60)
print("STEP 3A: Simple Linear Regression (area → price)")
print("=" * 60)

X_simple = df_model[["area"]]
y = df_model["price"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y, test_size=0.2, random_state=42)

slr = LinearRegression()
slr.fit(X_train_s, y_train_s)
y_pred_s = slr.predict(X_test_s)

mae_s  = mean_absolute_error(y_test_s, y_pred_s)
mse_s  = mean_squared_error(y_test_s, y_pred_s)
rmse_s = np.sqrt(mse_s)
r2_s   = r2_score(y_test_s, y_pred_s)

print(f"\n  Coefficient (slope) : {slr.coef_[0]:,.2f}")
print(f"  Intercept           : {slr.intercept_:,.2f}")
print(f"\n  MAE  : {mae_s:,.0f}")
print(f"  MSE  : {mse_s:,.0f}")
print(f"  RMSE : {rmse_s:,.0f}")
print(f"  R²   : {r2_s:.4f}")


STEP 3A: Simple Linear Regression (area → price)

  Coefficient (slope) : 425.73
  Intercept           : 2,512,254.26

  MAE  : 1,474,748
  MSE  : 3,675,286,604,768
  RMSE : 1,917,104
  R²   : 0.2729


In [5]:
# ============================================================
# STEP 4 — Part B: Multiple Linear Regression (all features)
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Multiple Linear Regression (all features)")
print("=" * 60)

feature_cols = [c for c in df_model.columns if c != "price"]
X_multi = df_model[feature_cols]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_m)
X_test_scaled  = scaler.transform(X_test_m)

mlr = LinearRegression()
mlr.fit(X_train_scaled, y_train_m)
y_pred_m = mlr.predict(X_test_scaled)

mae_m  = mean_absolute_error(y_test_m, y_pred_m)
mse_m  = mean_squared_error(y_test_m, y_pred_m)
rmse_m = np.sqrt(mse_m)
r2_m   = r2_score(y_test_m, y_pred_m)

print(f"\n  MAE  : {mae_m:,.0f}")
print(f"  MSE  : {mse_m:,.0f}")
print(f"  RMSE : {rmse_m:,.0f}")
print(f"  R²   : {r2_m:.4f}")

# Coefficients table
coef_df = pd.DataFrame({
    "Feature":     feature_cols,
    "Coefficient": mlr.coef_
}).sort_values("Coefficient", key=abs, ascending=False)
print("\n📌 Feature Coefficients (standardized — bigger abs = more influence):")
print(coef_df.to_string(index=False))



STEP 4: Multiple Linear Regression (all features)

  MAE  : 979,680
  MSE  : 1,771,751,116,594
  RMSE : 1,331,071
  R²   : 0.6495

📌 Feature Coefficients (standardized — bigger abs = more influence):
         Feature   Coefficient
       bathrooms 523153.382843
            area 519288.130105
 airconditioning 362446.186466
         stories 348177.113610
        prefarea 266661.049149
         parking 192786.985891
        basement 188462.048566
furnishingstatus 158183.270169
 hotwaterheating 150570.028279
        mainroad 128115.927978
       guestroom  89357.646208
        bedrooms  58690.918498


In [6]:
# ============================================================
# STEP 5 — Plot 1: Simple Regression Line
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Generating Plots")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter + regression line
x_range = np.linspace(X_simple["area"].min(), X_simple["area"].max(), 200).reshape(-1, 1)
y_line  = slr.predict(x_range)

axes[0].scatter(X_test_s, y_test_s, color="#3498db", alpha=0.6, s=40, label="Actual")
axes[0].plot(x_range, y_line, color="#e74c3c", linewidth=2, label="Regression Line")
axes[0].set_title("Simple Linear Regression\nArea vs Price", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Area (sq ft)")
axes[0].set_ylabel("Price (₹)")
axes[0].legend()

# Residual plot for simple regression
residuals_s = y_test_s - y_pred_s
axes[1].scatter(y_pred_s, residuals_s, color="#9b59b6", alpha=0.6, s=40)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("Residual Plot — Simple Regression", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predicted Price")
axes[1].set_ylabel("Residuals")

plt.suptitle("Simple Linear Regression Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("lr_01_simple_regression.png", bbox_inches="tight")
plt.close()
print("✅ Saved: lr_01_simple_regression.png")


STEP 5: Generating Plots
✅ Saved: lr_01_simple_regression.png


In [7]:
# ============================================================
# STEP 6 — Plot 2: Actual vs Predicted (Multiple Regression)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test_m, y_pred_m, color="#2ecc71", alpha=0.6, s=40)
min_val = min(y_test_m.min(), y_pred_m.min())
max_val = max(y_test_m.max(), y_pred_m.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect Prediction")
axes[0].set_title("Actual vs Predicted Price\n(Multiple Regression)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Actual Price (₹)")
axes[0].set_ylabel("Predicted Price (₹)")
axes[0].legend()

# Residual plot for multiple regression
residuals_m = y_test_m - y_pred_m
axes[1].scatter(y_pred_m, residuals_m, color="#e67e22", alpha=0.6, s=40)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("Residual Plot — Multiple Regression", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predicted Price")
axes[1].set_ylabel("Residuals")

plt.suptitle("Multiple Linear Regression Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("lr_02_multiple_regression.png", bbox_inches="tight")
plt.close()
print("✅ Saved: lr_02_multiple_regression.png")


✅ Saved: lr_02_multiple_regression.png


In [8]:
# ============================================================
# STEP 7 — Plot 3: Feature Coefficients Bar Chart
# ============================================================
plt.figure(figsize=(10, 6))
colors = ["#e74c3c" if c > 0 else "#3498db" for c in coef_df["Coefficient"]]
bars = plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="white", alpha=0.85)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Feature Coefficients — Multiple Linear Regression\n(Standardized — positive = increases price, negative = decreases)",
          fontsize=12, fontweight="bold")
plt.xlabel("Coefficient Value")
for bar, val in zip(bars, coef_df["Coefficient"]):
    plt.text(val + (50000 if val >= 0 else -50000), bar.get_y() + bar.get_height()/2,
             f"{val:,.0f}", va="center", fontsize=9,
             ha="left" if val >= 0 else "right")
plt.tight_layout()
plt.savefig("lr_03_feature_coefficients.png", bbox_inches="tight")
plt.close()
print("✅ Saved: lr_03_feature_coefficients.png")

✅ Saved: lr_03_feature_coefficients.png


In [9]:
# ============================================================
# STEP 8 — Plot 4: Model Comparison + Metrics Summary
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Metrics comparison bar chart
metrics = ["MAE", "RMSE", "R²"]
simple_vals  = [mae_s/1e6,  rmse_s/1e6,  r2_s]
multi_vals   = [mae_m/1e6,  rmse_m/1e6,  r2_m]

x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, simple_vals, w, label="Simple LR", color="#3498db", alpha=0.85)
axes[0].bar(x + w/2, multi_vals,  w, label="Multiple LR", color="#e74c3c", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_title("Model Comparison\n(MAE & RMSE in millions ₹)", fontsize=12, fontweight="bold")
axes[0].legend()
for i, (sv, mv) in enumerate(zip(simple_vals, multi_vals)):
    axes[0].text(i - w/2, sv + 0.01, f"{sv:.3f}", ha="center", fontsize=9)
    axes[0].text(i + w/2, mv + 0.01, f"{mv:.3f}", ha="center", fontsize=9)

# Distribution of residuals
axes[1].hist(residuals_m / 1e6, bins=30, color="#9b59b6", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_title("Distribution of Residuals\n(Multiple Regression)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Residual (₹ millions)")
axes[1].set_ylabel("Frequency")

plt.suptitle("Model Evaluation Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("lr_04_model_evaluation.png", bbox_inches="tight")
plt.close()
print("✅ Saved: lr_04_model_evaluation.png")

✅ Saved: lr_04_model_evaluation.png


In [10]:
# ============================================================
# STEP 9 — Multicollinearity Check (VIF)
# ============================================================
print("\n" + "=" * 60)
print("STEP 9: Multicollinearity Check (Correlation Matrix)")
print("=" * 60)

plt.figure(figsize=(12, 9))
corr = df_model.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={"size": 9})
plt.title("Correlation Matrix — All Features\n(check for multicollinearity)",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("lr_05_correlation_matrix.png", bbox_inches="tight")
plt.close()
print("✅ Saved: lr_05_correlation_matrix.png")


STEP 9: Multicollinearity Check (Correlation Matrix)
✅ Saved: lr_05_correlation_matrix.png


In [11]:
# ============================================================
# STEP 10 — Final Summary
# ============================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"""
┌──────────────────────────────────────────────────────────┐
│                   MODEL PERFORMANCE                      │
├────────────────────┬───────────────┬─────────────────────┤
│ Metric             │  Simple LR    │  Multiple LR        │
├────────────────────┼───────────────┼─────────────────────┤
│ MAE  (₹)           │ {mae_s:>13,.0f} │ {mae_m:>19,.0f} │
│ RMSE (₹)           │ {rmse_s:>13,.0f} │ {rmse_m:>19,.0f} │
│ R² Score           │ {r2_s:>13.4f} │ {r2_m:>19.4f} │
│ Features Used      │ {'1 (area)':>13} │ {'12 (all)':>19} │
└────────────────────┴───────────────┴─────────────────────┘
""")
print(f"  📌 Top feature: '{coef_df.iloc[0]['Feature']}' with coefficient {coef_df.iloc[0]['Coefficient']:,.0f}")
print(f"  📌 Multiple LR improved R² by {(r2_m - r2_s):.4f} over Simple LR")
print("\n🎉 Task 3 Complete! All 5 plots saved.")



FINAL SUMMARY

┌──────────────────────────────────────────────────────────┐
│                   MODEL PERFORMANCE                      │
├────────────────────┬───────────────┬─────────────────────┤
│ Metric             │  Simple LR    │  Multiple LR        │
├────────────────────┼───────────────┼─────────────────────┤
│ MAE  (₹)           │     1,474,748 │             979,680 │
│ RMSE (₹)           │     1,917,104 │           1,331,071 │
│ R² Score           │        0.2729 │              0.6495 │
│ Features Used      │      1 (area) │            12 (all) │
└────────────────────┴───────────────┴─────────────────────┘

  📌 Top feature: 'bathrooms' with coefficient 523,153
  📌 Multiple LR improved R² by 0.3766 over Simple LR

🎉 Task 3 Complete! All 5 plots saved.
